In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("..\\data\\WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.dtypes

In [ ]:
df.isnull().sum()

In [ ]:
list(df.columns)

In [ ]:
df["Churn"].value_counts()

In [ ]:
print(df["Churn"].value_counts(normalize=True) * 100)

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

In [ ]:
display(df.head())


In [ ]:
df.describe()

In [ ]:
df.loc[df["MonthlyCharges"] < 18.50, ["MonthlyCharges", "Churn"]]

In [ ]:
print(pd.crosstab(
    df["Contract"],
    df["Churn"],
    normalize="index"
) * 100)

In [ ]:
print(df.groupby("Churn")["tenure"].describe())

In [ ]:
print(df.groupby("Churn")["MonthlyCharges"].mean())

In [ ]:
df.info()

In [ ]:
print(df["TotalCharges"].head(10))
print("Data type:", df["TotalCharges"].dtype)

In [ ]:
for col in df.select_dtypes(include=["object", "str"]).columns:
    print(f"\n{col}:")
    print(df[col].unique())

In [ ]:
print(df["TotalCharges"].dtypes)

In [ ]:
df.isnull().sum()

In [ ]:
df["TotalCharges"] = df["TotalCharges"].str.strip()

In [ ]:
df[df["TotalCharges"].str.strip() == ""]


In [ ]:
print((df["TotalCharges"].str.strip() == "").sum())

In [ ]:
display(df["TotalCharges"].dtype)

In [ ]:
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

print("Data type:", df["TotalCharges"].dtype)
print("Missing values:", df["TotalCharges"].isnull().sum())

In [ ]:
display(
    df.loc[
        df["TotalCharges"].isnull(),
        ["tenure", "MonthlyCharges", "TotalCharges", "Churn"]
    ]
)

In [ ]:
df[df["TotalCharges"].isnull()]["tenure"].value_counts()

In [ ]:
df["TotalCharges"] = df["TotalCharges"].fillna(0)

In [ ]:
df["TotalCharges"].isna().sum()

In [ ]:
y = df["Churn"]

In [ ]:
X = df.drop(columns=["customerID", "Churn"])

In [ ]:
print(X.head())

In [ ]:
print(y.head())

In [ ]:
print("Numerical columns:")
print(X.select_dtypes(include=["number"]).columns.tolist())

print("\nCategorical columns:")
print(X.select_dtypes(include=["object", "category"]).columns.tolist())

In [ ]:
print(f"X.shape: {X.shape}")
print(f"y.shape: {y.shape}")

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

numeric_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numeric_transformer = StandardScaler()

categorical_transformer = OneHotEncoder(
    handle_unknown="ignore"
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
    
)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000))
    ]
)

In [ ]:

model.fit(X_train, y_train)


In [ ]:
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import confusion_matrix

y_pred = model.predict(X_test)

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred, labels=["No", "Yes"]))

In [ ]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=200,
            random_state=42
        ))
    ]
)


In [ ]:

rf_model.fit(X_train, y_train)


In [ ]:

from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import confusion_matrix

rf_pred = rf_model.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, rf_pred))

print("\nClassification Report:")
print(classification_report(y_test, rf_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, rf_pred, labels=["No", "Yes"]))


In [ ]:

print("Model Comparison")
print("-" * 40)

print(
    f"Logistic Regression Accuracy: "
    f"{accuracy_score(y_test, y_pred):.4f}"
)

print(
    f"Random Forest Accuracy: "
    f"{accuracy_score(y_test, rf_pred):.4f}"
)


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

models = {
    "Logistic Regression": y_pred,
    "Random Forest": rf_pred
}

for name, predictions in models.items():
    print(f"\n{name}")
    print("Churn Precision:", round(
        precision_score(y_test, predictions, pos_label="Yes"), 3
    ))
    print("Churn Recall:", round(
        recall_score(y_test, predictions, pos_label="Yes"), 3
    ))
    print("Churn F1-score:", round(
        f1_score(y_test, predictions, pos_label="Yes"), 3
    ))

In [ ]:
# Get the fitted preprocessing step
fitted_preprocessor = model.named_steps["preprocessor"]

# Get the fitted Logistic Regression classifier
classifier = model.named_steps["classifier"]

# Retrieve names of all transformed features
feature_names = fitted_preprocessor.get_feature_names_out()

# Retrieve the learned model coefficients
coefficients = classifier.coef_[0]

# Combine them into a DataFrame
importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients
})

# Sort by coefficient
importance_df = importance_df.sort_values(
    by="Coefficient",
    ascending=False
)

display(importance_df.head(10))

In [ ]:
display(importance_df.tail(10).sort_values("Coefficient"))

In [ ]:
import joblib

joblib.dump(model, "churn_model.joblib")

print("Model saved successfully!")